In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

In [ ]:
df = pd.read_csv("train.csv")
df.shape

In [ ]:
df.head()

## EDA


In [ ]:
df.info()

In [ ]:
# Missing Values
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_percentage = (missing / len(df) * 100).round(2)

print("\nMissing values")
print(pd.DataFrame({"count": missing, "percentage": missing_percentage}).head(20))

In [ ]:
# Visualise top 20 missing columns
plt.figure(figsize=(10, 5))
missing_percentage.head(20).plot(kind="bar", color="#EF9F27", edgecolor="white")
plt.title("Top 20 columns with missing values (%)")
plt.ylabel("% missing")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("missing_values.png", dpi=150)
plt.show()

In [ ]:
# Categorical: NaN means "no feature" (e.g. no pool, no garage)
none_cols = [
    "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
    "MasVnrType", "MSSubClass"
]
for col in none_cols:
    if col in df.columns:
        df[col] = df[col].fillna("None")


In [ ]:
# Numerical: NaN means 0 (e.g. no garage = 0 cars)
zero_cols = [
    "GarageYrBlt", "GarageArea", "GarageCars",
    "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
    "BsmtFullBath", "BsmtHalfBath", "MasVnrArea"
]
for col in zero_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)

In [ ]:
# Fill remaining categoricals with mode
cat_cols = df.select_dtypes(include="object").columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [ ]:
# Fill remaining numericals with median
num_cols = df.select_dtypes(include=[np.number]).columns
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

In [ ]:
print(f"\nMissing values remaining: {df.isnull().sum().sum()}")

In [ ]:
# TARGET VARIABLE ANALYSIS
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Raw distribution — will be right-skewed
axes[0].hist(df["SalePrice"], bins=50, color="#378ADD", edgecolor="white")
axes[0].set_title("SalePrice — raw (skewed)")
axes[0].set_xlabel("Price")

# Log-transformed — much more normal
axes[1].hist(np.log1p(df["SalePrice"]), bins=50, color="#1D9E75", edgecolor="white")
axes[1].set_title("SalePrice — log1p (normal-ish)")
axes[1].set_xlabel("log(Price)")

plt.tight_layout()
plt.savefig("target_distribution.png", dpi=150)
plt.show()


In [ ]:
# Apply log transform — this is your actual target going forward
df["SalePrice_log"] = np.log1p(df["SalePrice"])
print(f"\nSkewness before log: {df['SalePrice'].skew():.2f}")
print(f"Skewness after log:  {df['SalePrice_log'].skew():.2f}")

In [ ]:
# GrLivArea vs SalePrice — classic outlier check
plt.figure(figsize=(8, 5))
plt.scatter(df["GrLivArea"], df["SalePrice"], alpha=0.4, color="#378ADD")
plt.xlabel("GrLivArea (sq ft)")
plt.ylabel("SalePrice")
plt.title("GrLivArea vs SalePrice — spot the outliers")
plt.tight_layout()
plt.savefig("outliers_scatter.png", dpi=150)
plt.show()

In [ ]:
# Remove the two well-known outliers (huge area, very low price)
df = df[~((df["GrLivArea"] > 4000) & (df["SalePrice"] < 200000))]
print(f"\nShape after removing outliers: {df.shape}")

In [ ]:
# ORRELATION HEATMAP (top numeric features)
top_corr_features = (
    df.select_dtypes(include=[np.number])
    .corr()["SalePrice"]
    .abs()
    .sort_values(ascending=False)
    .head(15)
    .index
)

plt.figure(figsize=(10, 8))
sns.heatmap(
    df[top_corr_features].corr(),
    annot=True, fmt=".2f", cmap="Blues",
    square=True, linewidths=0.5
)
plt.title("Correlation matrix — top 15 features")
plt.tight_layout()
plt.savefig("correlation_heatmap.png", dpi=150)
plt.show()

In [ ]:
df.to_csv("train_cleaned.csv", index=False)
print("\nSaved: train_cleaned.csv")

## Feature Engineering

In [ ]:
df = pd.read_csv("train_cleaned.csv")
print(f"Loaded: {df.shape}")

In [ ]:
# Total square footage — single most powerful engineered feature
df["TotalSF"] = df["TotalBsmtSF"] + df["1stFlrSF"] + df["2ndFlrSF"]

In [ ]:
# House age and remodel signals
df["HouseAge"]       = df["YrSold"] - df["YearBuilt"]
df["RemodAge"]       = df["YrSold"] - df["YearRemodAdd"]
df["IsRemodeled"]    = (df["YearRemodAdd"] != df["YearBuilt"]).astype(int)
df["IsNewHouse"]     = (df["YrSold"] == df["YearBuilt"]).astype(int)

In [ ]:
# Bathroom totals
df["TotalBaths"] = (
    df["FullBath"] + df["HalfBath"] * 0.5 +
    df["BsmtFullBath"] + df["BsmtHalfBath"] * 0.5
)

In [ ]:
# Porch total area
df["TotalPorchSF"] = (
    df["OpenPorchSF"] + df["EnclosedPorch"] +
    df["3SsnPorch"] + df["ScreenPorch"]
)

In [ ]:
# Has garage / pool / basement flags
df["HasGarage"]   = (df["GarageArea"] > 0).astype(int)
df["HasPool"]     = (df["PoolArea"] > 0).astype(int)
df["HasBasement"] = (df["TotalBsmtSF"] > 0).astype(int)
df["Has2ndFloor"] = (df["2ndFlrSF"] > 0).astype(int)

In [ ]:
# Quality × condition interaction
df["OverallScore"] = df["OverallQual"] * df["OverallCond"]

# Living area per room
df["AreaPerRoom"] = df["GrLivArea"] / (df["TotRmsAbvGrd"] + 1)

In [ ]:
print("New features created:")
new_feats = [
    "TotalSF", "HouseAge", "RemodAge", "IsRemodeled", "IsNewHouse",
    "TotalBaths", "TotalPorchSF", "HasGarage", "HasPool",
    "HasBasement", "Has2ndFloor", "OverallScore", "AreaPerRoom"
]
print(df[new_feats].describe().T[["mean", "std", "min", "max"]])

In [ ]:
# Ordinal mappings — preserve meaningful order
ordinal_maps = {
    "ExterQual":  {"Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "ExterCond":  {"Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "BsmtQual":   {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "BsmtCond":   {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "HeatingQC":  {"Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "KitchenQual":{"Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "FireplaceQu":{"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "GarageQual": {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "GarageCond": {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "GarageFinish":{"None": 0, "Unf": 1, "RFn": 2, "Fin": 3},
    "BsmtExposure":{"None": 0, "No": 1, "Mn": 2, "Av": 3, "Gd": 4},
    "BsmtFinType1":{"None": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6},
    "BsmtFinType2":{"None": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6},
    "Functional":  {"Sal": 1, "Sev": 2, "Maj2": 3, "Maj1": 4, "Mod": 5, "Min2": 6, "Min1": 7, "Typ": 8},
    "LotShape":    {"IR3": 1, "IR2": 2, "IR1": 3, "Reg": 4},
    "LandSlope":   {"Sev": 1, "Mod": 2, "Gtl": 3},
    "PavedDrive":  {"N": 0, "P": 1, "Y": 2},
    "PoolQC":      {"None": 0, "Fa": 1, "TA": 2, "Gd": 3, "Ex": 4},
    "Fence":       {"None": 0, "MnWw": 1, "GdWo": 2, "MnPrv": 3, "GdPrv": 4},
}

for col, mapping in ordinal_maps.items():
    if col in df.columns:
        df[col] = df[col].map(mapping).fillna(0).astype(int)


In [ ]:
# One-hot encode remaining low-cardinality categoricals
cat_cols = df.select_dtypes(include="object").columns.tolist()
high_card = [c for c in cat_cols if df[c].nunique() > 10]
low_card  = [c for c in cat_cols if df[c].nunique() <= 10]

print(f"\nOne-hot encoding {len(low_card)} low-cardinality columns...")
df = pd.get_dummies(df, columns=low_card, drop_first=True)

# Label encode remaining high-cardinality columns
print(f"Label encoding {len(high_card)} high-cardinality columns: {high_card}")
le = LabelEncoder()
for col in high_card:
    df[col] = le.fit_transform(df[col].astype(str))

In [ ]:
# dropping columns
drop_cols = ["Id", "SalePrice"]   # keep SalePrice_log as target
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

print(f"\nFinal feature matrix shape: {df.shape}")
print(f"Target: SalePrice_log  |  Features: {df.shape[1] - 1}")

In [ ]:
df.to_csv("train_engineered.csv", index=False)
print("\nSaved: train_engineered.csv")

# Model


In [ ]:
df = pd.read_csv("train_engineered.csv")

X = df.drop(columns=["SalePrice_log"])
y = df["SalePrice_log"]

print(f"Features: {X.shape[1]}  |  Samples: {X.shape[0]}")

In [ ]:
models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=300,      # number of trees
        max_depth=15,          # how deep each tree can grow
        min_samples_leaf=2,    # minimum samples required at a leaf node
        n_jobs=-1,             # use all CPU cores
        random_state=42
    ),
    "XGBoost": xgb.XGBRegressor(
        n_estimators=1000,
        learning_rate=0.05,    # smaller = slower but more accurate
        max_depth=4,           # shallower trees reduce overfitting
        subsample=0.8,         # use 80% of rows per tree
        colsample_bytree=0.8,  # use 80% of features per tree
        min_child_weight=3,
        reg_alpha=0.005,       # L1 regularisation
        reg_lambda=1.0,        # L2 regularisation
        random_state=42,
        verbosity=0
    ),
}


In [ ]:
#    CROSS-VALIDATION
#    5-fold CV — gives a reliable estimate of real-world performance
#    Metric: RMSE on log prices (lower is better)
#    Random Forest:  expect ~0.14–0.16
#    XGBoost:        expect ~0.115–0.125
# ─────────────────────────────────────────────
kf = KFold(n_splits=5, shuffle=True, random_state=42)
results = {}

print("\n── Cross-validation results (5-fold RMSE) ──")
for name, model in models.items():
    scores = np.sqrt(-cross_val_score(
        model, X, y,
        scoring="neg_mean_squared_error",
        cv=kf, n_jobs=-1
    ))
    results[name] = scores
    print(f"  {name:<18}  mean={scores.mean():.4f}  std={scores.std():.4f}")

In [ ]:
names  = list(results.keys())
means  = [results[n].mean() for n in names]
stds   = [results[n].std()  for n in names]
colors = ["#378ADD", "#1D9E75"]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(names, means, yerr=stds, capsize=5,
              color=colors, edgecolor="white", width=0.45)
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.001,
            f"{mean:.4f}", ha="center", va="bottom", fontsize=11)
ax.set_ylabel("RMSE (log scale)")
ax.set_title("Random Forest vs XGBoost — 5-fold CV RMSE (lower is better)")
ax.set_ylim(0, max(means) * 1.25)
plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150)
plt.show()

In [ ]:
best_name = min(results, key=lambda n: results[n].mean())
print(f"\nBest model: {best_name}  (RMSE={results[best_name].mean():.4f})")

best_model = models[best_name]
best_model.fit(X, y)

In [ ]:
feat_df = pd.DataFrame({
    "feature":    X.columns,
    "importance": best_model.feature_importances_
}).sort_values("importance", ascending=False).head(20)

plt.figure(figsize=(9, 6))
plt.barh(feat_df["feature"][::-1], feat_df["importance"][::-1],
         color="#378ADD", edgecolor="white")
plt.xlabel("Feature importance")
plt.title(f"Top 20 features — {best_name}")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150)
plt.show()

print("\nTop 10 most important features:")
print(feat_df.head(10).to_string(index=False))

In [ ]:
y_pred_log = best_model.predict(X)
y_pred     = np.expm1(y_pred_log)   # convert back to actual dollars
y_actual   = np.expm1(y)

plt.figure(figsize=(7, 6))
plt.scatter(y_actual, y_pred, alpha=0.3, color="#378ADD", s=15)
plt.plot([y_actual.min(), y_actual.max()],
         [y_actual.min(), y_actual.max()],
         "r--", lw=1.5, label="Perfect prediction")
plt.xlabel("Actual price ($)")
plt.ylabel("Predicted price ($)")
plt.title(f"Predicted vs Actual — {best_name}")
plt.legend()
plt.tight_layout()
plt.savefig("predicted_vs_actual.png", dpi=150)
plt.show()

In [ ]:
results_df = pd.DataFrame({
    "actual":    y_actual.values,
    "predicted": y_pred,
    "error_$":   (y_pred - y_actual.values).round(0)
}).head(10)

print("\nSample predictions (first 10 rows):")
print(results_df.to_string(index=False))

In [57]:
test = pd.read_csv("test.csv")
test_ids = test["Id"]

# ── Step 1: same null imputation as Phase 1 ──
none_cols = [
    "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
    "MasVnrType", "MSSubClass"
]
zero_cols = [
    "GarageYrBlt", "GarageArea", "GarageCars",
    "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
    "BsmtFullBath", "BsmtHalfBath", "MasVnrArea"
]
for col in none_cols:
    if col in test.columns: test[col] = test[col].fillna("None")
for col in zero_cols:
    if col in test.columns: test[col] = test[col].fillna(0)
for col in test.select_dtypes(include="object").columns:
    test[col] = test[col].fillna(test[col].mode()[0])
for col in test.select_dtypes(include=[np.number]).columns:
    test[col] = test[col].fillna(test[col].median())

# ── Step 2: same engineered features as Phase 2 ──
test["TotalSF"]      = test["TotalBsmtSF"] + test["1stFlrSF"] + test["2ndFlrSF"]
test["HouseAge"]     = test["YrSold"] - test["YearBuilt"]
test["RemodAge"]     = test["YrSold"] - test["YearRemodAdd"]
test["IsRemodeled"]  = (test["YearRemodAdd"] != test["YearBuilt"]).astype(int)
test["IsNewHouse"]   = (test["YrSold"] == test["YearBuilt"]).astype(int)
test["TotalBaths"]   = (test["FullBath"] + test["HalfBath"] * 0.5 +
                        test["BsmtFullBath"] + test["BsmtHalfBath"] * 0.5)
test["TotalPorchSF"] = (test["OpenPorchSF"] + test["EnclosedPorch"] +
                        test["3SsnPorch"] + test["ScreenPorch"])
test["HasGarage"]    = (test["GarageArea"] > 0).astype(int)
test["HasPool"]      = (test["PoolArea"] > 0).astype(int)
test["HasBasement"]  = (test["TotalBsmtSF"] > 0).astype(int)
test["Has2ndFloor"]  = (test["2ndFlrSF"] > 0).astype(int)
test["OverallScore"] = test["OverallQual"] * test["OverallCond"]
test["AreaPerRoom"]  = test["GrLivArea"] / (test["TotRmsAbvGrd"] + 1)

# ── Step 3: same ordinal encoding as Phase 2 ──
ordinal_maps = {
    "ExterQual":   {"Po":1,"Fa":2,"TA":3,"Gd":4,"Ex":5},
    "ExterCond":   {"Po":1,"Fa":2,"TA":3,"Gd":4,"Ex":5},
    "BsmtQual":    {"None":0,"Po":1,"Fa":2,"TA":3,"Gd":4,"Ex":5},
    "BsmtCond":    {"None":0,"Po":1,"Fa":2,"TA":3,"Gd":4,"Ex":5},
    "HeatingQC":   {"Po":1,"Fa":2,"TA":3,"Gd":4,"Ex":5},
    "KitchenQual": {"Po":1,"Fa":2,"TA":3,"Gd":4,"Ex":5},
    "FireplaceQu": {"None":0,"Po":1,"Fa":2,"TA":3,"Gd":4,"Ex":5},
    "GarageQual":  {"None":0,"Po":1,"Fa":2,"TA":3,"Gd":4,"Ex":5},
    "GarageCond":  {"None":0,"Po":1,"Fa":2,"TA":3,"Gd":4,"Ex":5},
    "GarageFinish":{"None":0,"Unf":1,"RFn":2,"Fin":3},
    "BsmtExposure":{"None":0,"No":1,"Mn":2,"Av":3,"Gd":4},
    "BsmtFinType1":{"None":0,"Unf":1,"LwQ":2,"Rec":3,"BLQ":4,"ALQ":5,"GLQ":6},
    "BsmtFinType2":{"None":0,"Unf":1,"LwQ":2,"Rec":3,"BLQ":4,"ALQ":5,"GLQ":6},
    "Functional":  {"Sal":1,"Sev":2,"Maj2":3,"Maj1":4,"Mod":5,"Min2":6,"Min1":7,"Typ":8},
    "LotShape":    {"IR3":1,"IR2":2,"IR1":3,"Reg":4},
    "LandSlope":   {"Sev":1,"Mod":2,"Gtl":3},
    "PavedDrive":  {"N":0,"P":1,"Y":2},
    "PoolQC":      {"None":0,"Fa":1,"TA":2,"Gd":3,"Ex":4},
    "Fence":       {"None":0,"MnWw":1,"GdWo":2,"MnPrv":3,"GdPrv":4},
}
for col, mapping in ordinal_maps.items():
    if col in test.columns:
        test[col] = test[col].map(mapping).fillna(0).astype(int)

# ── Step 4: same one-hot + label encoding as Phase 2 ──
from sklearn.preprocessing import LabelEncoder
cat_cols  = test.select_dtypes(include="object").columns.tolist()
high_card = [c for c in cat_cols if test[c].nunique() > 10]
low_card  = [c for c in cat_cols if test[c].nunique() <= 10]
test = pd.get_dummies(test, columns=low_card, drop_first=True)
le = LabelEncoder()
for col in high_card:
    test[col] = le.fit_transform(test[col].astype(str))

# ── Step 5: align test columns to exactly match training columns ──
# Add any missing columns (some dummies may not appear in test)
for col in X.columns:
    if col not in test.columns:
        test[col] = 0

# Keep only the columns the model was trained on, in the same order
test = test[X.columns]

# ── Step 6: predict and save ──
test_preds = np.expm1(best_model.predict(test))

submission = pd.DataFrame({"Id": test_ids, "SalePrice": test_preds})
submission.to_csv("submission.csv", index=False)
print(submission.head())
print(f"\nSubmission saved — {len(submission)} rows")

     Id      SalePrice
0  1461  126015.039062
1  1462  165808.968750
2  1463  187949.390625
3  1464  191456.703125
4  1465  177892.218750

Submission saved — 1459 rows
